# VarChat — Qwen2.5-14B QLoRA eğitimi (eğitilmiş 14B hücresi)

**Nasıl açılır ve çalıştırılır (Colab'ın sitesi):**
1. colab.research.google.com → **Dosya → Not defteri yükle** (File → Upload notebook) → bilgisayarındaki `fine_tune/f03_colab_train_14b.ipynb`.
2. **Çalışma zamanı → Çalışma zamanı türünü değiştir** (Runtime → Change runtime type) → **A100 GPU**; "Yüksek RAM" (High-RAM) seçeneği görünürse aç → Kaydet.
   A100 boşta değilse **L4** seç: yine bf16, yalnız eğitim ~2,5-3 saate çıkar. **T4 seçme:** bf16 desteklemez, defter 0. bölümde durur.
3. **Çalışma zamanı → Tümünü çalıştır** (Run all). 2. bölümde Google Drive izni sorulur: kendi hesabınla izin ver.
   İzin eğitimden ÖNCE sorulur; eğitim bittiğinde başında beklemen gerekmez.

**Süre:** A100'de ~1-1,5 saat (taban modeli indirme ~5-10 dk, eğitim ~55 dk, doğrulama ölçümleri birkaç dk).
Bu sürede tarayıcı sekmesi açık kalsın ve bilgisayar uykuya geçmesin (Windows: Ayarlar → Sistem → Güç → uyku "Hiçbir zaman").
**Kayıt:** eğitim sunucunun diskinde yapılır; bitince adaptör ve eğitim kayıtları Drive'da **`varchat-14b/varchat-lora-14b/`** klasörüne
kopyalanır (6. bölüm). 7B'nin Drive klasörlerine (`varchat-lora-v2`, `varchat-olcum`, `varchat-gguf`) dokunulmaz.
**Bitince:** **Çalışma zamanı → Bağlantıyı kes ve çalışma zamanını sil** (birim harcamasın), sonra e12'ye geç (en alttaki "Sonraki adım").

Bu defter `f01_colab_train.ipynb`'nin kopyasıdır ve tarif aynıdır: QLoRA (NF4, çift niceleme, bf16), r=16, alpha=32, dropout 0,05,
yedi doğrusal modül türü, MAX_LEN 4608, batch 1 × birikim 8, 3 tur, öğrenme oranı 2e-4, kosinüs, 5 ısınma adımı, paged AdamW 8-bit,
kayıp yalnız cevap tokenlarında, doğrulama kaybı en düşük olan tur saklanır. Veri: `fine_tune/egitim_verisi.jsonl` (438 eğitim, 63 doğrulama).

**f01'den farkları**
1. Taban model **Qwen/Qwen2.5-14B-Instruct**. Eğitim sunucunun diskindeki `/content/varchat-lora-14b` klasörüne yazılır, bitince Drive'da
   **`varchat-14b/varchat-lora-14b`** klasörüne kopyalanır (f01 doğrudan Drive'daki `varchat-lora-v2` klasörüne yazıyordu; ona dokunulmaz).
2. `TUR` ve `TOHUM` en üstte birer ayar: 3 ve 42, yani 7B'nin değerleri (7B'de tohum kodda yazmıyordu, kütüphanenin varsayılanı 42 kullanıldı).
   Tohum burada LoRA ek matrislerinin başlangıç değerlerini de belirler (7B'de bu değerler tohumsuz üretilmişti; iki modelin ek matrisleri zaten farklı boyutta).
3. bf16 zorunlu: kart desteklemiyorsa defter durur (f01 bu durumda fp16'ya geçiyordu).
4. Eğitim verisinin parmak izi (sha256) 7B'yi eğiten dosyayla karşılaştırılır. Kodun commit kimliği, kartın adı, RAM, en yüksek GPU belleği ve Liger durumu `ortam.json` dosyasına yazılır.
5. Drive'da ya da sunucuda bitmiş bir adaptör varsa defter durur (üzerine yazmaz). Yarım kalmış bir koşunun checkpoint klasörleri silinir, eğitim baştan başlar.
6. f01'in 7. bölümü (canlı PubMed ile göz kontrolü) yok. Ölçüm, `evaluation/e12_colab_14b.ipynb` defterinde uygulamanın kendi hattıyla yapılır.
7. Drive'a kopyalanan her dosyanın parmak izi `drive_kopyasi.json` dosyasına yazılır; bu dosya en son yazılır, yani varsa kopya tamamdır.
   e12 adaptörü okumadan önce bu listeye bakar.

## Ayar

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"                     # tek bilincli degisiklik: 7B yerine 14B
KAYIT_DIZINI = '/content/varchat-lora-14b'                 # egitim sunucunun diskinde (hizli; Drive'daki kisa kopmalardan etkilenmez)
DRIVE_KOK = '/content/drive/MyDrive/varchat-14b'           # bu hucrenin Drive klasoru (7B'nin klasorlerinden ayri)
DRIVE_ADAPTOR = f'{DRIVE_KOK}/varchat-lora-14b'            # bitince adaptor + kayitlar buraya kopyalanir; e12 buradan okur
TUR = 3                                                    # num_train_epochs (7B: 3)
TOHUM = 42                                                 # rastgelelik tohumu (7B: 42, kutuphane varsayilani)
VERI_SHA256 = "7b974d73b1ed421a3aa84918c5d2cda1db9f008dc8d7e78aedca9fba66a1afb0"   # 7B'yi egiten egitim_verisi.jsonl (commit 255ecd4; GitHub'daki LF surum)

ESKI_KLASORLER = ("varchat-lora", "varchat-lora-v2", "varchat-olcum", "varchat-gguf")   # 7B'nin Drive klasorleri
for yol in (KAYIT_DIZINI, DRIVE_KOK, DRIVE_ADAPTOR):
    assert yol.rstrip("/").split("/")[-1] not in ESKI_KLASORLER, f"7B'nin klasorlerine yazilmaz: {yol}. Ayar hucresinde yeni bir ad ver."

## 0) Sunucu kontrolü: bf16, RAM, disk

e12 ayrı bir oturumda çalışacağı için yalnız bu defterin ihtiyacına bakılır: kart **gerçek bf16** desteklemeli (A100, L4; T4 desteklemez,
bugünkü PyTorch T4'te bf16'yı yalnız taklit eder) ve en az ~22 GB belleği olmalı (14B eğitimi A100'de en çok 19,8 GB kullandı; T4 15 GB).
RAM en az 25 GB, boş disk en az ~40 GB (taban model 29,5 GB + checkpoint'ler ~2 GB + paketler). Yetmiyorsa eğitimden önce durur:
**Çalışma zamanı → Çalışma zamanı türünü değiştir** → A100 ya da L4 → **Tümünü çalıştır**.

In [ ]:
!nvidia-smi
import os, shutil
# Bellek parcalanmasini azaltir; kart ilk kullanilmadan ONCE ayarlanmali (f01'de oyleydi; 3. bolumdeki ayni satir burada gec kalirdi)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from huggingface_hub.constants import HF_HUB_CACHE
assert torch.cuda.is_available(), "GPU yok: Calisma zamani → Calisma zamani turunu degistir → A100 GPU, sonra Tumunu calistir."
KART = torch.cuda.get_device_name(0)
KART_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9)
YETENEK = torch.cuda.get_device_capability(0)   # gercek bf16: 8.0 ve ustu (A100 8.0, L4 8.9); T4 7.5
assert YETENEK >= (8, 0), (f"Bu kart {KART} (yetenek {YETENEK[0]}.{YETENEK[1]}): gercek bf16 yok, bellegi de yetmez. "
                           "Colab A100/L4 yerine bunu vermis. Calisma zamani → Calisma zamani turunu degistir → A100 (yoksa L4) → Kaydet → Tumunu calistir.")
assert KART_GB >= 22, f"{KART} bellegi {KART_GB} GB: 14B egitimi ~20 GB ister. Calisma zamani turunu A100 (yoksa L4) yap, Tumunu calistir."
RAM_GB = round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9)
DISK_GB = round(shutil.disk_usage('/content').free / 1e9)
TABAN_INDIRILMIS = os.path.isdir(os.path.join(HF_HUB_CACHE, "models--" + MODEL_ID.replace("/", "--")))   # ayni oturumda ikinci kosu
GEREKEN_DISK = 10 if TABAN_INDIRILMIS else 40
print(f"kart: {KART} ({KART_GB} GB) | RAM: {RAM_GB} GB | bos disk: {DISK_GB} GB (gereken {GEREKEN_DISK}) | bf16: {torch.cuda.is_bf16_supported()}")
assert torch.cuda.is_bf16_supported(), \
    f"{KART} bf16 desteklemiyor (T4 gibi). 7B bf16 ile egitildi; Calisma zamani turunu A100 ya da L4 yap, Tumunu calistir."
assert RAM_GB >= 25, f"RAM {RAM_GB} GB: Calisma zamani turunu A100 ya da L4 yap (Yuksek RAM aciksa daha iyi), Tumunu calistir."
assert DISK_GB >= GEREKEN_DISK, f"Bos disk {DISK_GB} GB: ~{GEREKEN_DISK} GB gerekir. Durumu Claude'a yaz."

## 1) Kütüphaneler (7B eğitimindeki sürümler)

In [ ]:
# 7B egitimiyle ayni surumler (25 Eylul 2026'da dogrulanmis; tez tekrar uretilebilirligi icin sabit)
!pip install -q "transformers==5.17.0" "peft==0.21.0" "bitsandbytes==0.50.2" "accelerate==1.15.0" "datasets==5.0.1" "liger-kernel==0.8.3"
# Surumler paket kayitlarindan okunur (liger_kernel modulunde __version__ yok). torch, Colab'in o gunku surumudur (7B: 2.11.0+cu130).
import importlib.metadata as paket
import torch
SURUMLER = {ad: paket.version(ad) for ad in ("transformers", "peft", "bitsandbytes", "accelerate", "datasets", "liger-kernel", "torch")}
SURUMLER["cuda"] = torch.version.cuda
print(SURUMLER)

## 2) Projeyi klonla, veriyi doğrula, Drive'a bağlan, kayıt klasörünü hazırla

Repo GitHub'dan sunucuya gelir. Eğitim verisinin parmak izi (satır sonları LF'ye çevrilerek) 7B'yi eğiten dosyayla aynı değilse defter durur.
Sonra Drive izni sorulur (açılan pencerede kendi Google hesabını seç, izin ver). Drive'da bitmiş bir 14B adaptörü varsa defter durur:
eğitim zaten yapılmış demektir, e12'ye geç.

In [ ]:
import os, glob, shutil, hashlib

# Tum proje (kod + veri). Ayni oturumda ikinci kez calistirilirsa klon yerine guncelle.
if not os.path.exists('/content/varchat'):
    !git clone https://github.com/atu-ce/varchat.git /content/varchat
else:
    !git -C /content/varchat pull -q
%cd /content/varchat
KOD = !git -C /content/varchat log --oneline -1 --no-color
print("kod:", KOD[0])

DATA_PATH = "/content/varchat/fine_tune/egitim_verisi.jsonl"
assert os.path.exists(DATA_PATH), f"Bulunamadi: {DATA_PATH}"
# Satir sonlari LF'ye cevrilip izlenir: Windows kopyasi (CRLF) ile Colab kopyasi (LF) ayni izi verir; icerikteki tek harf farki yine yakalanir
VERI_IZI = hashlib.sha256(open(DATA_PATH, "rb").read().replace(b"\r\n", b"\n")).hexdigest()
assert VERI_IZI == VERI_SHA256, f"egitim_verisi.jsonl 7B'yi egiten dosyayla ayni degil (sha256 {VERI_IZI[:12]}...)"
print("veri:", DATA_PATH, "| parmak izi 7B'yi egiten dosyayla ayni")

# Drive: izin simdi sorulur (egitimden once). Bitmis bir 14B adaptoru varsa uzerine yazilmaz.
from google.colab import drive
drive.mount('/content/drive')
assert not os.path.exists(f"{DRIVE_ADAPTOR}/adapter_model.safetensors"), \
    (f"Drive'da {DRIVE_ADAPTOR} icinde bir adaptor var; uzerine yazilmaz. Egitim zaten yapilmis: e12'ye gec. "
     "(drive_kopyasi.json yoksa kopya yarim kalmis olabilir; durumu Claude'a yaz.)")
os.makedirs(DRIVE_KOK, exist_ok=True)
DRIVE_BOS_GB = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
print(f"Drive: {DRIVE_KOK} | bos yer ~{DRIVE_BOS_GB:.1f} GB")
if DRIVE_BOS_GB < 1:
    print("UYARI: Drive'da 1 GB'tan az bos yer gorunuyor; 6. bolumdeki kopya (~0,3 GB) sigmayabilir. Egitimden once Drive'da yer ac.")

# Kayit dizini (sunucunun diski; yalniz yeni klasor)
os.makedirs(KAYIT_DIZINI, exist_ok=True)
assert not os.path.exists(f"{KAYIT_DIZINI}/adapter_model.safetensors"), \
    f"{KAYIT_DIZINI} icinde bitmis bir adaptor var: egitim bu oturumda zaten bitmis. Yeniden egitme; yalniz 6. bolumu (Drive'a kopya) calistir."
for eski in sorted(glob.glob(f"{KAYIT_DIZINI}/checkpoint-*")):     # yarim kalmis onceki kosu (yalniz bu klasorde)
    print("yarim kalmis checkpoint siliniyor:", eski)
    shutil.rmtree(eski)
print("kayit ->", KAYIT_DIZINI, "| bitince ->", DRIVE_ADAPTOR)

## 3) Modeli 4-bit (QLoRA) yükle

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # parcalanmayi azalt (asil etkili olan 0. bolumdeki)

import torch
from packaging import version
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Liger Kernel: fused cross-entropy -> uzun dizide logit bellegini dusurur. Uyumsuzsa egitimi durdurma (durum ortam.json'a yazilir).
try:
    from liger_kernel.transformers import apply_liger_kernel_to_qwen2
    apply_liger_kernel_to_qwen2()
    LIGER = True
    print("Liger kernel: aktif")
except Exception as e:
    LIGER = False
    print("Liger kernel uygulanamadi, standart cekirdekle devam:", type(e).__name__, e)

BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
assert BF16, "bf16 yok (0. bolume bak)"
COMPUTE_DTYPE = torch.bfloat16
print("GPU:", KART, "| bf16:", BF16)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# transformers >= 4.56 'dtype' ister ('torch_dtype' 5.x'te yalnizca uyari ile calisir)
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56.0") else "torch_dtype"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", **{DTYPE_KW: COMPUTE_DTYPE},
)
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
)
model.config.use_cache = False

set_seed(TOHUM)   # LoRA ek matrislerinin baslangic degerleri bu tohumla uretilir (f01'de bu satir yok; ust aciklama 2. madde)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # beklenen: ~68,8 milyon egitilebilir ayar (7B'de 40,4 milyon)
print(f"GPU bellegi (model yuklu): {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 4) Veriyi hazırla: doğrulama seti + chat şablonu + cevap-only etiketler

f01 ile aynı. Bölme **gen bazlı** ve veri dosyasında sabit: 14 genin (AKT1, DNMT3A, HFE, HRAS, IDH2, MYD88, LRRK2, MEFV, CALR, ALDH2, RHO, PCSK9, NF1, SCN1A)
tüm örnekleri doğrulama setinde. Qwen'in chat şablonu uygulanır ve **yalnızca son cevabın tokenları etiketlenir**; öncesi `-100` olur.
Qwen2.5'in 7B ve 14B modelleri aynı tokenizer'ı kullanır (sözlük 152.064); en uzun örnek yine ~4146 token olmalı (aşağıda yazdırılır).

In [ ]:
import collections
from datasets import load_dataset

ds = load_dataset("json", data_files=DATA_PATH, split="train")   # alanlar: messages, varyant, bolme, tur

ds_train = ds.filter(lambda x: x["bolme"] == "train")
ds_val   = ds.filter(lambda x: x["bolme"] == "val")
VAL_VARYANTLAR = sorted(set(ds_val["varyant"]))
print("doğrulama varyantları:", VAL_VARYANTLAR)
print("eğitim örneği:", len(ds_train), "| doğrulama örneği:", len(ds_val))
print("eğitim tür dağılımı:", dict(collections.Counter(ds_train["tur"])))
assert (len(ds_train), len(ds_val)) == (438, 63), "eğitim/doğrulama sayısı 7B'dekiyle (438/63) aynı değil"

MAX_LEN = 4608   # 7B ile ayni; asan ornek olursa asagidaki hucre hata verir (cevabi kirpmayiz)

In [ ]:
from transformers import DataCollatorForSeq2Seq

def tokenle(ornek):
    """Chat şablonunu uygular; system+user kısmının etiketlerini -100 yapar (kayıp yalnızca cevapta)."""
    msgs = ornek["messages"]
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    full   = tok.apply_chat_template(msgs,      tokenize=False, add_generation_prompt=False)
    assert full.startswith(prompt), "şablon öneki uyuşmuyor"
    p_ids = tok(prompt, add_special_tokens=False).input_ids
    f_ids = tok(full,   add_special_tokens=False).input_ids
    assert f_ids[:len(p_ids)] == p_ids, "önek tokenizasyonu uyuşmuyor"
    if len(f_ids) > MAX_LEN:
        raise ValueError(f"örnek MAX_LEN'i aşıyor ({len(f_ids)} token): {ornek['varyant']} — kaynakları kısalt, cevabı kırpma")
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    return {"input_ids": f_ids, "attention_mask": [1] * len(f_ids), "labels": labels}

ds_train_tok = ds_train.map(tokenle, remove_columns=ds_train.column_names)
ds_val_tok   = ds_val.map(tokenle,   remove_columns=ds_val.column_names)

# Kontrol: kaybin hesaplandigi (cevap) tokenlarin payi ve en uzun ornek
toplam = sum(len(x["input_ids"]) for x in ds_train_tok)
hedef  = sum(sum(1 for l in x["labels"] if l != -100) for x in ds_train_tok)
EN_UZUN = max(len(x["input_ids"]) for d in (ds_train_tok, ds_val_tok) for x in d)
print(f"eğitim tokenları: {toplam:,} | kayıp hesaplanan (cevap) tokenlar: {hedef:,} = %{100*hedef/toplam:.1f} | en uzun örnek: {EN_UZUN} token")

collator = DataCollatorForSeq2Seq(tok, label_pad_token_id=-100, padding=True)
print("hazir:", ds_train_tok, ds_val_tok)

## 5) Eğitim (QLoRA)

Eğitim öncesi ve her tur sonunda **doğrulama kaybı** (görülmemiş genlerde) ölçülür. A100'de 14B ile ~55 dk
(7B aynı tarifle 26 dk sürmüştü; 14B'de token başına iş yaklaşık iki kat; L4'te ~2,5-3 saat). Adapter her turda, kayıp geçmişi ve
ortam bilgisi sonda sunucunun diskine (`KAYIT_DIZINI`) yazılır; 6. bölüm hepsini Drive'a kopyalar.

In [ ]:
from transformers import TrainingArguments, Trainer
import inspect, json, datetime

gecerli = set(inspect.signature(TrainingArguments.__init__).parameters)
eval_anahtar = "eval_strategy" if "eval_strategy" in gecerli else "evaluation_strategy"   # surum farki

istenen = dict(
    output_dir=KAYIT_DIZINI,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=TUR,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,          # dogrulama kaybi en dusuk turu tut (ezber baslarsa son tur degil)
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=BF16, fp16=not BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
    seed=TOHUM,                           # 7B'de yazilmamisti; varsayilan deger de 42
)
istenen[eval_anahtar] = "epoch"

kwargs  = {k: v for k, v in istenen.items() if k in gecerli}
atlanan = [k for k in istenen if k not in gecerli]
if atlanan:
    print("Bu surumde desteklenmeyen (atlandi, varsayilan kullanilacak):", atlanan)

args = TrainingArguments(**kwargs)
trainer = Trainer(model=model, args=args,
                  train_dataset=ds_train_tok, eval_dataset=ds_val_tok, data_collator=collator,
                  processing_class=tok)    # 5.x adi ('tokenizer=' artik yok)

torch.cuda.reset_peak_memory_stats()
print("eğitim ÖNCESİ doğrulama kaybı:", round(trainer.evaluate()["eval_loss"], 4))
trainer.train()

with open(f"{KAYIT_DIZINI}/log_history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=1)
with open(f"{KAYIT_DIZINI}/dogrulama_varyantlari.json", "w") as f:
    json.dump(VAL_VARYANTLAR, f, ensure_ascii=False, indent=1)
with open(f"{KAYIT_DIZINI}/surumler.json", "w") as f:
    json.dump(SURUMLER, f, indent=1)
trainer.state.save_to_json(f"{KAYIT_DIZINI}/trainer_state_son.json")   # en iyi checkpoint ve tur bilgisi
ORTAM = {"tarih": datetime.datetime.now().isoformat(timespec="minutes"), "model": MODEL_ID, "tur": TUR, "tohum": TOHUM,
         "kod": KOD[0], "veri_sha256": VERI_IZI, "kart": KART, "kart_gb": KART_GB, "ram_gb": RAM_GB,
         "en_yuksek_gpu_bellegi_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1), "liger": LIGER, "bf16": BF16,
         "max_len": MAX_LEN, "en_uzun_ornek": EN_UZUN, "en_iyi_checkpoint": trainer.state.best_model_checkpoint,
         "en_iyi_eval_loss": trainer.state.best_metric}
with open(f"{KAYIT_DIZINI}/ortam.json", "w") as f:
    json.dump(ORTAM, f, ensure_ascii=False, indent=1)
print(json.dumps(ORTAM, ensure_ascii=False, indent=1))
print("kaydedildi ->", KAYIT_DIZINI, "(log_history, dogrulama_varyantlari, surumler, trainer_state_son, ortam)")

In [ ]:
trainer.save_model(KAYIT_DIZINI)      # LoRA adapter (en dusuk dogrulama kaybi olan tur; ~275 MB) -> sunucunun diski
tok.save_pretrained(KAYIT_DIZINI)
print("kaydedildi ->", KAYIT_DIZINI, "|", sorted(os.listdir(KAYIT_DIZINI)))

## 6) Drive'a kopyala

Sunucunun diski çalışma zamanı silinince gider; adaptör ve eğitim kayıtları Drive'da **`varchat-14b/varchat-lora-14b/`** klasörüne kopyalanır
(checkpoint klasörleri hariç, ~290 MB): adaptör, adaptör ayarı, tokenizer, kayıp geçmişi, doğrulama varyantları, sürümler, eğitim durumu,
ortam bilgisi, eğitim ayarları (`training_args.bin`). Her dosyanın parmak izi en son `drive_kopyasi.json` dosyasına yazılır.
Sonra Drive bağlantısı kapatılır: bekleyen yazmalar bu sırada Drive'a gönderilir (sonraki bölümler Drive kullanmaz).

In [ ]:
import json, hashlib, datetime

def sha256(yol):
    h = hashlib.sha256()
    with open(yol, "rb") as f:
        for parca in iter(lambda: f.read(1 << 24), b""):
            h.update(parca)
    return h.hexdigest()

dosyalar = sorted(a for a in os.listdir(KAYIT_DIZINI) if os.path.isfile(f"{KAYIT_DIZINI}/{a}"))   # checkpoint klasorleri haric
for ad in ("adapter_model.safetensors", "adapter_config.json", "log_history.json", "ortam.json", "training_args.bin"):
    assert ad in dosyalar, f"{KAYIT_DIZINI} icinde {ad} yok: 5. bolum bitmemis"
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')            # bu hucre tek basina yeniden calistirilirsa
os.makedirs(DRIVE_ADAPTOR, exist_ok=True)
izler = {}
for ad in dosyalar:
    shutil.copy2(f"{KAYIT_DIZINI}/{ad}", f"{DRIVE_ADAPTOR}/{ad}")
    izler[ad] = sha256(f"{KAYIT_DIZINI}/{ad}")
    assert sha256(f"{DRIVE_ADAPTOR}/{ad}") == izler[ad], f"{ad} Drive'a eksik kopyalandi; bu hucreyi yeniden calistir"
    print(f"  {ad:34} {os.path.getsize(f'{KAYIT_DIZINI}/{ad}') / 1e6:8.1f} MB")
# En son yazilir: bu dosya varsa kopya tamamdir (e12 adaptoru okumadan once buna bakar)
with open(f"{DRIVE_ADAPTOR}/drive_kopyasi.json", "w") as f:
    json.dump({"tarih": datetime.datetime.now().isoformat(timespec="minutes"), "kaynak": KAYIT_DIZINI, "kod": KOD[0],
               "dosyalar": izler}, f, indent=1)
print("Drive'a kopyalandı ->", DRIVE_ADAPTOR, f"({len(izler)} dosya + drive_kopyasi.json)")
try:
    drive.flush_and_unmount()                # bekleyen yazmalari Drive'a gonderir, sonra baglantiyi kapatir
    print("Drive'a gönderildi; Drive bağlantısı kapatıldı.")
except Exception as e:
    print("UYARI: Drive bağlantısı düzgün kapatılamadı:", e,
          "\nÇalışma zamanını silmeden önce drive.google.com'da varchat-14b/varchat-lora-14b/drive_kopyasi.json dosyasını gör.")

## 7) Hızlı test — doğrulama setinden bir varyant (eğitimde GÖRÜLMEMİŞ)

f01 ile aynı göz kontrolü: Türkçe mi, her cümlede [n] var mı, kaynak dışına çıkıyor mu. Öğretmenin cevabıyla yan yana basılır. Sayısal ölçüm değildir.

In [ ]:
model.config.use_cache = True
model.eval()

def uret(system, user, max_new_tokens=600):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

ornek = ds_val[0]                      # egitimde olmayan varyant
m = {x["role"]: x["content"] for x in ornek["messages"]}
print("VARYANT:", ornek["varyant"], "| SORU:", m["user"], "\n")
print("MODEL:\n", uret(m["system"], m["user"]))
print("\nÖĞRETMEN:\n", m["assistant"])

### Sonraki adım

1. drive.google.com'da `varchat-14b/varchat-lora-14b/` klasöründe `adapter_model.safetensors` ve `drive_kopyasi.json` olduğunu gör.
2. **Çalışma zamanı → Bağlantıyı kes ve çalışma zamanını sil** (A100 açık kaldıkça birim yakar).
3. **`evaluation/e12_colab_14b.ipynb`** defterini aynı yolla aç (Dosya → Not defteri yükle), çalışma zamanı türü **L4 GPU**
   ("Yüksek RAM" görünürse aç) → **Tümünü çalıştır**. e12 adaptörü Drive'dan okur.

**Notlar**
- `OutOfMemory` olursa: `MAX_LEN`'i DÜŞÜRME (cevabı kırpar). L4'teysen A100 seç; A100'de olduysa durumu Claude'a yaz.
- `Liger kernel uygulanamadı` yazarsa eğitim yine çalışır (bellek biraz artar); durum `ortam.json`'a yazılır, tezde not edilir.
- Bağlantı koparsa: eğitim bitmeden koptuysa defteri yeniden **Tümünü çalıştır** (eğitim baştan başlar).
  Drive'da `drive_kopyasi.json` varsa eğitimi tekrarlama, e12'ye geç.
- Eğitim kayıtlarını bilgisayarına almak (acil değil): Drive'da `varchat-lora-14b` klasörüne sağ tık → İndir. Kayıt dosyaları projede
  `fine_tune/results/v2-14b/` klasörüne (7B'ninkiler `fine_tune/results/v2/` içinde), adaptör `models/varchat-lora-14b/` klasörüne (git'e girmez).